# 4. Hourly Demand Modeling â€” LightGBM

This notebook trains and evaluates the hourly LightGBM model using the feature-engineered dataset from `transformed_dataset_post_FE.xlsx`.

**Goals:**
1. Load and clean the final feature set
2. Train/test split by date (temporal, not random)
3. Train LightGBM with early stopping
4. Evaluate vs original model baseline (MAE 1.638, RMSE 3.818, sMAPE 49.19%)
5. Analyze feature importance
6. Visualize predictions

In [32]:
# Cell 1: Imports
import json
import pickle
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

## Cell 2: Load and preprocess data

In [61]:
# Cell 2: Load and preprocess data

df = pd.read_excel("../data/processed/transformed_dataset_post_FE.xlsx")
print(f"Loaded: {df.shape}")

# Ensure datetime
df["programmed_departure_time"] = pd.to_datetime(df["programmed_departure_time"])

# Drop duplicate hour column
if "programmed_departure_hour" in df.columns and "hour" in df.columns:
    df = df.drop(columns=["programmed_departure_hour"])
    print("Dropped duplicate: programmed_departure_hour")

# Fill NaNs in lag/rolling features with 0
lag_roll_cols = [c for c in df.columns if "lag_" in c or "roll_" in c]
df[lag_roll_cols] = df[lag_roll_cols].fillna(0.0)
print(f"Filled NaNs in {len(lag_roll_cols)} lag/roll columns")

# Verify zero NaNs
assert df.isna().sum().sum() == 0, "Unexpected NaNs remaining!"
print("Zero NaNs across entire dataset")

# Save to parquet
df.to_parquet("../data/processed/hourly_features_test.parquet", index=False)
print("Saved parquet")

Loaded: (305521, 42)
Dropped duplicate: programmed_departure_hour
Filled NaNs in 18 lag/roll columns
Zero NaNs across entire dataset
Saved parquet


## Cell 3: Define feature set

In [62]:
# Cell 3: Original feature set — no extras, no experiments
# Original 30 features minus 3 rejected (days_to_quincena, is_quincena, days_since_first_record)

FEATURE_COLS = [
    "plant_code",
    "hour",
    "day_of_week",
    "month_of_year",
    "day_of_month",
    "year",
    "is_weekend",
    "is_sunday",
    "was_open",
    "hour_sin",
    "hour_cos",
    "day_week_sin",
    "day_week_cos",
    "month_sin",
    "month_cos",
    "days_since_last_open",
    "volume_per_remission_7d_avg",
    "is_holiday",
    "volume_m3_lag_24h",
    "volume_m3_lag_48h",
    "volume_m3_lag_1w",
    "volume_m3_roll_mean_24h",
    "volume_m3_roll_mean_48h",
    "volume_m3_roll_mean_1w",
    "volume_m3_roll_std_24h",
    "volume_m3_roll_std_48h",
    "volume_m3_roll_std_1w",
]

TARGET_COL = "volume_m3"

print(f"Features: {len(FEATURE_COLS)}")

missing = [c for c in FEATURE_COLS if c not in df.columns]
assert len(missing) == 0, f"Missing columns: {missing}"
print("All feature columns present.")

Features: 27
All feature columns present.


## Cell 4: Train / test split

In [63]:
# Temporal split: train on historical data, test on recent data
# Using same cutoff as original model for fair comparison
TEST_CUTOFF_DATE = "2025-01-01"

train_mask = df["programmed_departure_time"] < TEST_CUTOFF_DATE
train_df = df[train_mask].copy()
test_df = df[~train_mask].copy()

print(f"Train: {len(train_df):,} rows ({train_df['programmed_departure_time'].min()} to {train_df['programmed_departure_time'].max()})")
print(f"Test:  {len(test_df):,} rows ({test_df['programmed_departure_time'].min()} to {test_df['programmed_departure_time'].max()})")
print(f"Train %: {len(train_df)/len(df)*100:.1f}%")
print(f"Test %:  {len(test_df)/len(df)*100:.1f}%")

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL]

Train: 236,623 rows (2020-02-04 05:00:00 to 2024-12-31 23:00:00)
Test:  68,898 rows (2025-01-01 00:00:00 to 2026-04-24 11:00:00)
Train %: 77.4%
Test %:  22.6%


## Cell 5: Train LightGBM

In [64]:
# Cell 5: Train LightGBM — original hyperparameters (no regularization experiments)

LGB_PARAMS = {
    "objective": "regression",
    "metric": "mae",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "random_state": 42,
}

NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 50

print("Training LightGBM...")
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

model = lgb.train(
    LGB_PARAMS,
    train_data,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[train_data, valid_data],
    valid_names=["train", "valid"],
    callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True)],
)

print(f"\nBest iteration: {model.best_iteration}")

Training LightGBM...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[230]	train's l1: 2.37296	valid's l1: 2.43321

Best iteration: 230


## Cell 6: Evaluate

In [65]:
def smape(y_true, y_pred):
    """Symmetric MAPE."""
    return (200 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred))).mean()

# Predict on test set
y_pred = model.predict(X_test, num_iteration=model.best_iteration)
y_pred = np.clip(y_pred, 0, None)  # Volume cannot be negative

errors = y_test - y_pred
mae = np.abs(errors).mean()
rmse = np.sqrt((errors ** 2).mean())
smape_val = smape(y_test, y_pred)

print("\n========== TEST METRICS ==========")
print(f"MAE:   {mae:.4f} m3/hr")
print(f"RMSE:  {rmse:.4f} m3/hr")
print(f"sMAPE: {smape_val:.2f}%")
print("\n========== ORIGINAL BASELINE ==========")
print("MAE:   1.6380 m3/hr")
print("RMSE:  3.8177 m3/hr")
print("sMAPE: 49.19%")
print("\n========== DELTA ==========")
print(f"MAE:   {mae - 1.6380:+.4f}")
print(f"RMSE:  {rmse - 3.8177:+.4f}")
print(f"sMAPE: {smape_val - 49.19:+.2f} pp")


========== TEST METRICS ==========
MAE:   2.4252 m3/hr
RMSE:  6.2732 m3/hr
sMAPE: 152.64%

========== ORIGINAL BASELINE ==========
MAE:   1.6380 m3/hr
RMSE:  3.8177 m3/hr
sMAPE: 49.19%

========== DELTA ==========
MAE:   +0.7872
RMSE:  +2.4555
sMAPE: +103.45 pp


## Cell 7: Per-plant metrics

In [66]:
# Per-plant breakdown
test_eval = test_df.copy()
test_eval["predicted"] = y_pred
test_eval["error"] = errors

print("\nPer-plant MAE:")
print("-" * 40)
for plant in sorted(test_eval["plant_code"].unique()):
    sub = test_eval[test_eval["plant_code"] == plant]
    plant_mae = sub["error"].abs().mean()
    plant_rmse = np.sqrt((sub["error"] ** 2).mean())
    print(f"Plant {plant}: MAE={plant_mae:.4f}, RMSE={plant_rmse:.4f}, n={len(sub)}")

# Compare to original per-plant MAE
print("\nOriginal per-plant MAE:")
print("-" * 40)
original_mae = {510: 1.73, 511: 1.81, 512: 1.65, 514: 1.47, 515: 1.94, 710: 1.23}
for plant, orig in original_mae.items():
    sub = test_eval[test_eval["plant_code"] == plant]
    new_mae = sub["error"].abs().mean()
    print(f"Plant {plant}: {orig:.2f} â†’ {new_mae:.2f} ({new_mae - orig:+.2f})")


Per-plant MAE:
----------------------------------------
Plant 510: MAE=2.6815, RMSE=6.1211, n=11482
Plant 511: MAE=2.6386, RMSE=7.3803, n=11483
Plant 512: MAE=2.3227, RMSE=5.1223, n=11484
Plant 514: MAE=2.1087, RMSE=5.0281, n=11483
Plant 515: MAE=2.5089, RMSE=6.7833, n=11483
Plant 710: MAE=2.2911, RMSE=6.8302, n=11483

Original per-plant MAE:
----------------------------------------
Plant 510: 1.73 â†’ 2.68 (+0.95)
Plant 511: 1.81 â†’ 2.64 (+0.83)
Plant 512: 1.65 â†’ 2.32 (+0.67)
Plant 514: 1.47 â†’ 2.11 (+0.64)
Plant 515: 1.94 â†’ 2.51 (+0.57)
Plant 710: 1.23 â†’ 2.29 (+1.06)


## Cell 8: Feature importance

In [58]:
# Gain-based feature importance
importance = pd.DataFrame({
    "feature": model.feature_name(),
    "gain": model.feature_importance(importance_type="gain"),
    "split": model.feature_importance(importance_type="split"),
})
importance = importance.sort_values("gain", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=importance["gain"],
    y=importance["feature"],
    orientation="h",
    marker_color="#2ca02c"
))
fig.update_layout(
    title=f"Feature Importance (Gain) â€” {len(FEATURE_COLS)} features",
    xaxis_title="Gain",
    yaxis_title="",
    template="plotly_white",
    height=700,
    margin=dict(l=150)
)
fig.show()

print("\nTop 10 features by gain:")
print(importance.sort_values("gain", ascending=False)[["feature", "gain"]].head(10).to_string(index=False))


Top 10 features by gain:
               feature         gain
  days_since_last_open 3.396427e+07
     volume_m3_lag_24h 7.200167e+06
                  hour 6.311226e+06
consecutive_zero_hours 5.748365e+06
              hour_cos 5.137438e+06
     volume_m3_lag_48h 2.956821e+06
              hour_sin 2.900495e+06
       is_morning_ramp 2.601760e+06
           day_of_week 2.398901e+06
volume_m3_roll_mean_1w 1.714830e+06


## Cell 9: Prediction visualization

In [59]:
# Plot actual vs predicted for a sample week per plant
sample_plants = sorted(test_eval["plant_code"].unique())
sample_week = test_eval["programmed_departure_time"].max() - pd.Timedelta(days=7)
sample_df = test_eval[test_eval["programmed_departure_time"] >= sample_week].copy()

fig = make_subplots(
    rows=len(sample_plants), cols=1,
    subplot_titles=[f"Plant {p}" for p in sample_plants],
    shared_xaxes=True,
    vertical_spacing=0.04,
)

for i, plant in enumerate(sample_plants, 1):
    sub = sample_df[sample_df["plant_code"] == plant].sort_values("programmed_departure_time")
    fig.add_trace(
        go.Scatter(x=sub["programmed_departure_time"], y=sub["volume_m3"],
                   mode="lines", name=f"Actual {plant}", line=dict(color="#1f77b4")),
        row=i, col=1
    )
    fig.add_trace(
        go.Scatter(x=sub["programmed_departure_time"], y=sub["predicted"],
                   mode="lines", name=f"Predicted {plant}", line=dict(color="#ff7f0e", dash="dash")),
        row=i, col=1
    )

fig.update_layout(
    title=f"Actual vs Predicted â€” Last Week of Test Set ({sample_week.strftime('%Y-%m-%d')} onwards)",
    template="plotly_white",
    height=200 * len(sample_plants),
    showlegend=False,
)
fig.update_yaxes(title_text="m3/hr")
fig.show()

## Cell 10: Residual analysis

In [60]:
# Residuals by hour of day
test_eval["hour"] = test_eval["programmed_departure_time"].dt.hour
residual_by_hour = test_eval.groupby("hour")["error"].agg(["mean", "std"]).reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Mean Error by Hour", "Error Std Dev by Hour"])

fig.add_trace(
    go.Bar(x=residual_by_hour["hour"], y=residual_by_hour["mean"], marker_color="#d62728"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=residual_by_hour["hour"], y=residual_by_hour["std"], marker_color="#9467bd"),
    row=1, col=2
)

fig.update_layout(template="plotly_white", height=400, showlegend=False)
fig.show()

print("\nMean error by hour (bias check):")
print(residual_by_hour.round(3).to_string(index=False))


Mean error by hour (bias check):
 hour   mean    std
    0 -0.047  0.134
    1 -0.035  0.503
    2 -0.060  0.254
    3 -0.018  3.054
    4 -0.060  0.310
    5 -0.199  1.270
    6 -0.249  6.315
    7  0.499 13.240
    8  0.282  8.993
    9 -0.117  8.641
   10  0.497  8.549
   11 -0.216  8.287
   12 -0.341  7.179
   13 -0.175  7.289
   14  0.004  7.185
   15 -0.028  6.482
   16 -0.414  5.508
   17  0.348  9.787
   18 -0.023  4.481
   19  0.438  9.159
   20 -0.047  0.613
   21 -0.041  0.130
   22  0.046  2.534
   23 -0.051  0.269


## Cell 11: Save artifacts (optional)

Saves to a **test directory** to avoid overwriting production artifacts.
Verify metrics before promoting to .


In [ ]:
# Save model, metrics, and validation CSV to TEST directory
# (Production artifacts in app/ are NOT overwritten)

TEST_MODEL_DIR = Path("../models/test")
TEST_OUTPUT_DIR = Path("../outputs/test_forecasts")
TEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = TEST_MODEL_DIR / "hourly_lgbm.pkl"
METRICS_PATH = TEST_MODEL_DIR / "metrics_hourly.json"
VALIDATION_PATH = TEST_OUTPUT_DIR / "validation_hourly.csv"

# Save model
with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)
print(f"Model saved: {MODEL_PATH}")

# Save metrics
metrics = {
    "mae_hourly_m3": round(float(mae), 4),
    "rmse_hourly_m3": round(float(rmse), 4),
    "smape_pct": round(float(smape_val), 2),
    "validation_rows": int(len(test_df)),
    "date_calculated": datetime.now().isoformat(),
    "test_cutoff_date": TEST_CUTOFF_DATE,
    "best_iteration": int(model.best_iteration),
    "by_plant": {},
}
for plant in sorted(test_eval["plant_code"].unique()):
    sub = test_eval[test_eval["plant_code"] == plant]
    metrics["by_plant"][str(int(plant))] = {
        "mae": round(float(sub["error"].abs().mean()), 4),
        "rmse": round(float(np.sqrt((sub["error"] ** 2).mean())), 4),
        "count": int(len(sub)),
    }

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved: {METRICS_PATH}")

# Save validation CSV
val_out = test_eval[["programmed_departure_time", "plant_code", "volume_m3", "predicted", "error"]].copy()
val_out.columns = ["hour_bucket", "ship_plant_code", "volume_m3", "predicted", "error"]
val_out.to_csv(VALIDATION_PATH, index=False)
print(f"Validation saved: {VALIDATION_PATH}")
